# 1. Package architecture

| Package | Role |
|---|---|
| `langchain` | streamlined high-level APIs |
| `langchain-core` | core abstractions and LCEL |
| `langchain-community` | community integrations |
| provider packages | OpenAI/Anthropic/Google/AWS/etc. integrations |
| `langgraph` | graph runtime/orchestration |
| `deepagents` | long-running agent harness |
| `langsmith` | tracing/evals/testing/deployment tooling |

**v1 change:** main `langchain` namespace was deliberately reduced. This made it look agent-centric, but retrieval/data components remain active across `langchain-core` and integration packages.

# 2. Installation

```bash
%pip install -U langchain langchain-core langchain-community langchain-text-splitters
%pip install -U langchain-openai langgraph deepagents langchain-mcp-adapters
```
Typical provider installs:

```bash
pip install -U langchain-openai
pip install -U langchain-anthropic
pip install -U langchain-google-genai
pip install -U langchain-aws
pip install -U langchain-huggingface
pip install -U langchain-ollama
```

In [2]:
CHAT_MODEL = "openai:gpt-5.4-mini"
EMBED_MODEL = "openai:text-embedding-3-small"

# 3. `Document` — retrieval data unit

In [1]:
from langchain_core.documents import Document

doc = Document(
    id="doc-001",
    page_content="LangChain provides composable building blocks.",
    metadata={"source": "course_notes", "chapter": 1},
)
print(doc.page_content)
print(doc.metadata)
print(doc.id)

LangChain provides composable building blocks.
{'source': 'course_notes', 'chapter': 1}
doc-001


# 4. Chat Models

In [3]:
from langchain.chat_models import init_chat_model
model = init_chat_model(CHAT_MODEL)
type(model)

langchain_openai.chat_models.base.ChatOpenAI

In [4]:
response = model.invoke("Explain semantic search in two sentences.")
print(type(response), response.content)

<class 'langchain_core.messages.ai.AIMessage'> Semantic search is a search method that looks for the meaning behind a query rather than just matching exact keywords. It uses techniques like embeddings and machine learning to find results that are conceptually related, even if they don’t share the same words.


In [5]:
responses = model.batch([
    "Define embeddings.",
    "Define vector store.",
    "Define retriever.",
])
for r in responses:
    print(r.content)

**Embeddings** are a way to represent objects—such as words, sentences, images, or users—as **vectors of numbers** so that **similar things end up close together** in that vector space.

### In simple terms
Instead of storing a word like `"cat"` as text, an embedding might represent it as something like:

`[0.12, -0.87, 0.44, ...]`

This numeric representation lets a computer compare meanings and relationships more effectively than using raw text.

### Why embeddings are useful
They capture patterns such as:
- **Semantic similarity**: “cat” and “dog” are closer than “cat” and “airplane”
- **Relationships**: some embeddings can reflect analogies like “king” relates to “queen”
- **Efficient machine learning input**: models can work with numbers more easily than strings

### Common types
- **Word embeddings**: represent individual words
- **Sentence embeddings**: represent whole sentences or paragraphs
- **Image embeddings**: represent images in a learned feature space
- **User/item embed

In [6]:
for chunk in model.stream("Explain LCEL briefly."):
    print(chunk.content, end="")

LCEL stands for **LangChain Expression Language**.

It’s a way to **compose LLM workflows as pipelines** using simple building blocks like prompts, models, parsers, retrievers, and functions.

### Core idea
Instead of writing a lot of custom orchestration code, you chain components together with operators like:

- `|` to pass output from one step to the next
- dictionaries to run steps in parallel
- helper methods for branching, retrying, streaming, etc.

### Example
```python
chain = prompt | llm | parser
```

This means:

1. format the prompt
2. send it to the language model
3. parse the result

### Why it’s useful
- **Readable**: easy to see the data flow
- **Composable**: small pieces can be reused
- **Supports streaming and async**
- **Works well with retrieval and tool use**

### In short
LCEL is LangChain’s “pipeline language” for building LLM apps in a clean, modular way.

```
invoke() → Use when you have one input and want one complete output; best for normal single-user requests like “summarize this text.”
batch() → Use when you have many independent inputs and want to process them together efficiently; best for cases like “summarize 100 documents/questions.”
stream() → Use when you want to receive the response progressively as it is generated instead of waiting for the full answer; best for chat UIs where users should see tokens/results immediately.
```

In [7]:
getattr(model, "profile", None)

{'name': 'GPT-5.4 mini',
 'release_date': '2026-03-17',
 'last_updated': '2026-03-17',
 'open_weights': False,
 'max_input_tokens': 400000,
 'max_output_tokens': 128000,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': False,
 'image_url_inputs': True,
 'pdf_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_choice': True,
 'tool_call_streaming': True,
 'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh']}

# 5. Messages

In [8]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
messages = [
    SystemMessage(content="You are a concise teacher."),
    HumanMessage(content="What is a vector store?"),
]
messages

[SystemMessage(content='You are a concise teacher.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is a vector store?', additional_kwargs={}, response_metadata={})]

In [9]:
msg = HumanMessage(content_blocks=[
    {"type": "text", "text": "Describe this image."},
    {"type": "image", "url": "https://example.com/image.jpg"},
])
msg.content_blocks

[{'type': 'text', 'text': 'Describe this image.'},
 {'type': 'image', 'url': 'https://example.com/image.jpg'}]

In [10]:
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    trim_messages
)

# Full chat history
history = [
    HumanMessage(content="Hello"),
    AIMessage(content="Hi! How can I help you?"),

    HumanMessage(content="What is LangChain?"),
    AIMessage(content="LangChain is a framework for building LLM applications."),

    HumanMessage(content="What is RAG?"),
    AIMessage(content="RAG combines retrieval with generation."),

    HumanMessage(content="Teach me RAG with an example.")
]


# Keep only the recent part of the conversation
trimmed_history = trim_messages(
    history,
    max_tokens=4,
    strategy="last",

    # For this simple demo, count each message as 1 unit
    token_counter=len,
)


print("FULL HISTORY")
print("-" * 50)

for message in history:
    print(type(message).__name__, ":", message.content)


print("\nTRIMMED HISTORY")
print("-" * 50)

for message in trimmed_history:
    print(type(message).__name__, ":", message.content)

FULL HISTORY
--------------------------------------------------
HumanMessage : Hello
AIMessage : Hi! How can I help you?
HumanMessage : What is LangChain?
AIMessage : LangChain is a framework for building LLM applications.
HumanMessage : What is RAG?
AIMessage : RAG combines retrieval with generation.
HumanMessage : Teach me RAG with an example.

TRIMMED HISTORY
--------------------------------------------------
AIMessage : LangChain is a framework for building LLM applications.
HumanMessage : What is RAG?
AIMessage : RAG combines retrieval with generation.
HumanMessage : Teach me RAG with an example.


# 6. Prompt Templates

In [11]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
prompt = PromptTemplate.from_template("Explain {topic} to a {level} learner.")
print(prompt.format(topic="embeddings", level="beginner"))

Explain embeddings to a beginner learner.


In [12]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI teacher. Explain at {level} level."),
    ("human", "{question}"),
])
chat_prompt.invoke({"level":"advanced", "question":"What is LCEL?"})

ChatPromptValue(messages=[SystemMessage(content='You are an AI teacher. Explain at advanced level.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is LCEL?', additional_kwargs={}, response_metadata={})])

In [ ]:
history_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a tutor."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
history_prompt.invoke({
    "history": [HumanMessage(content="I am learning LangChain."), AIMessage(content="Great.")],
    "question": "Teach retrievers.",
})

In [13]:
partial = chat_prompt.partial(level="expert")
partial.invoke({"question":"Explain MMR."})

ChatPromptValue(messages=[SystemMessage(content='You are an AI teacher. Explain at expert level.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain MMR.', additional_kwargs={}, response_metadata={})])

In [14]:
partial

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={'level': 'expert'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['level'], input_types={}, partial_variables={}, template='You are an AI teacher. Explain at {level} level.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])

```
PromptTemplate
    ↓
Plain text prompt


ChatPromptTemplate
    ↓
System + Human + AI messages


MessagesPlaceholder
    ↓
Insert existing message/history
inside ChatPromptTemplate
```

# 7. Output Parsers vs Native Structured Output

In [20]:
from langchain_core.output_parsers import StrOutputParser

string_parser = StrOutputParser()

response = model.invoke("What is RAG?")

result = string_parser.invoke(response)

print(result)

RAG stands for **Retrieval-Augmented Generation**.

It’s a technique used with AI/LLMs where the model:

1. **Retrieves** relevant information from an external source  
   - such as documents, a database, a website, or a vector store

2. **Augments** the prompt with that information

3. **Generates** an answer using both the retrieved context and its own language abilities

### Why use RAG?
- **More up-to-date answers** than relying on the model alone
- **Less hallucination** because answers are grounded in retrieved sources
- **Can use private or domain-specific data** without retraining the model

### Simple example
If you ask:  
> “What’s our company’s refund policy?”

A RAG system might:
- search your internal policy docs,
- pull the relevant section,
- then have the LLM answer based on that text.

### In short
RAG = **search relevant info first, then generate the response**.

If you want, I can also explain:
- how RAG works technically,
- how it differs from fine-tuning,
- or how 

In [21]:
chain = model | StrOutputParser()

result = chain.invoke("What is RAG?")

print(result)

RAG usually means **Retrieval-Augmented Generation**.

It’s an AI approach where a language model:
1. **Retrieves** relevant information from an external source, like a document database, files, or the web
2. **Augments** its context with that information
3. **Generates** an answer using both the retrieved data and its own model

### Why use RAG?
- Gives the model access to **up-to-date** or **private** information
- Reduces hallucinations by grounding answers in sources
- Lets you answer questions over your own documents without retraining the model

### Simple example
If you ask, “What is our company’s vacation policy?”, a RAG system might:
- search the HR handbook
- pull the relevant section
- have the model answer based on that text

### In short
RAG = **search + LLM answer**.

If you want, I can also explain:
- how RAG works technically
- a diagram of the pipeline
- how it differs from fine-tuning


In [22]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

json_parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Return the answer only as JSON.

        {format_instructions}
        """
    ),
    (
        "human",
        "Extract name and skills from: {text}"
    )
])

prompt = prompt.partial(
    format_instructions=json_parser.get_format_instructions()
)

chain = prompt | model | json_parser

result = chain.invoke({
    "text": "Sunny knows Python, LangChain and RAG"
})

print(result)
print(type(result))

{'name': 'Sunny', 'skills': ['Python', 'LangChain', 'RAG']}
<class 'dict'>


In [ ]:
from pydantic import BaseModel, Field
class Person(BaseModel):
    name: str = Field(description="Name of the person")
    age: int = Field(description="Age of the person")
    skills: list[str] = Field(
        description="Technical skills of the person"
    )

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
parser = PydanticOutputParser(
    pydantic_object=Person
)
print(parser.get_format_instructions()[:400])

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is n


In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Extract the information.

        {format_instructions}
        """
    ),
    (
        "human",
        "{text}"
    )
])

prompt = prompt.partial(
    format_instructions=parser.get_format_instructions()
)

In [ ]:
chain = prompt | model | parser

In [29]:
result = chain.invoke({
    "text":
    "Sunny is 31 years old and knows Python, LangChain and RAG."
})

print(result)

topic='Sunny' explanation='Sunny is 31 years old and knows Python, LangChain, and RAG.' difficulty='easy'


In [26]:
class CityDetails(BaseModel):
    city: str
    country: str
    population_note: str | None = None

structured_model = model.with_structured_output(CityDetails)

print(structured_model.invoke("Give details for Bengaluru."))

city='Bengaluru' country='India' population_note='One of India’s largest cities; population is widely reported at over 10 million in the metropolitan area, though exact figures vary by source and definition.'


## Tools and Manual Tool Calling

In [32]:
from langchain.tools import tool
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a*b
multiply.invoke({"a":6, "b":7})

42

In [33]:
tool_bound_model = model.bind_tools([multiply])

ai_msg = tool_bound_model.invoke("What is 12*8?")
print(ai_msg.tool_calls)

[{'name': 'multiply', 'args': {'a': 12, 'b': 8}, 'id': 'call_Q9ZMk68yssrgONKrpwyhlKoN', 'type': 'tool_call'}]


In [38]:

@tool
def search_knowledge_base(query: str) -> str:
    """Search internal knowledge base."""
    return f"Dummy result for query: {query}"

In [39]:
result = search_knowledge_base.invoke({
    "query": "What is RAG?"
})

print(result)

Dummy result for query: What is RAG?


In [40]:
from langchain.agents import create_agent
agent = create_agent(
    model=CHAT_MODEL,
    tools=[multiply, search_knowledge_base],
    system_prompt="You are helpful. Use tools when needed.",
)

result = agent.invoke({"messages":[{"role":"user","content":"What is 9*9?"}]})
print(result["messages"][-1].content)

81


In [ ]:
class ResearchAnswer(BaseModel):
    answer: str
    confidence: str
    sources_used: list[str]

structured_agent = create_agent(model=CHAT_MODEL, tools=[search_knowledge_base], response_format=ResearchAnswer)

In [ ]:
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
provider_strategy_agent = create_agent(model=CHAT_MODEL, tools=[], response_format=ProviderStrategy(ResearchAnswer))
tool_strategy_agent = create_agent(model=CHAT_MODEL, tools=[], response_format=ToolStrategy(ResearchAnswer))

ResearchAnswer → Defines the exact structured response schema: answer, confidence, and sources_used.

response_format=ResearchAnswer → Lets LangChain automatically choose the best structured-output method for that model.

ProviderStrategy(ResearchAnswer) → Forces LangChain to use the model provider’s native structured output feature.

ToolStrategy(ResearchAnswer) → Forces LangChain to generate structured output using tool/function calling.

provider_strategy_agent → Agent that uses provider-native structured output.

tool_strategy_agent → Agent that uses tool-calling-based structured output.

In [30]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime


# 1. Runtime context structure
@dataclass
class UserContext:
    user_id: str
    role: str


# 2. Tool
@tool
def show_profile(runtime: ToolRuntime[UserContext]) -> str:
    """Show the currently logged-in user's profile."""
    
    user_id = runtime.context.user_id
    role = runtime.context.role

    return f"User ID: {user_id}, Role: {role}"


# 3. Create agent
agent = create_agent(
    model=CHAT_MODEL,
    tools=[show_profile],
    context_schema=UserContext
)


# 4. Imagine this came from your login/authentication system
logged_in_user = {
    "id": "101",
    "role": "admin"
}


# 5. Invoke agent
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Show my profile"
            }
        ]
    },
    context=UserContext(
        user_id=logged_in_user["id"],
        role=logged_in_user["role"]
    )
)


# 6. Final answer
print(response["messages"][-1].content)

Your profile:

- User ID: 101
- Role: admin


In [ ]:
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model, after_model
from langgraph.runtime import Runtime


@before_model
def log_before(state: AgentState, runtime: Runtime):
    print("Model is about to run")
    return None


@after_model
def log_after(state: AgentState, runtime: Runtime):
    print("Model finished")
    return None


agent = create_agent(
    model=CHAT_MODEL,
    tools=[],
    middleware=[log_before, log_after]
)

response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is RAG?"}
    ]
})

User question
   ↓
before_model()
   ↓
LLM call
   ↓
after_model()
   ↓
Answer

In [41]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


@dataclass
class UserContext:
    role: str


@dynamic_prompt
def role_prompt(request: ModelRequest) -> str:

    role = request.runtime.context.role

    if role == "beginner":
        return "Explain everything in simple English."

    return "Explain everything technically and in detail."


agent = create_agent(
    model=CHAT_MODEL,
    tools=[],
    middleware=[role_prompt],
    context_schema=UserContext
)

In [42]:
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is RAG?"}
        ]
    },
    context=UserContext(role="beginner")
)

In [43]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest
from langchain.chat_models import init_chat_model


cheap_model = init_chat_model("openai:gpt-5.4-mini")
strong_model = init_chat_model("openai:gpt-5.4")


@wrap_model_call
def choose_model(request: ModelRequest, handler):

    message_count = len(request.state["messages"])

    if message_count > 10:
        selected_model = strong_model
    else:
        selected_model = cheap_model

    request = request.override(
        model=selected_model
    )

    return handler(request)

In [44]:
agent = create_agent(
    model=cheap_model,
    tools=[],
    middleware=[choose_model]
)

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Explain RAG in simple English."
        }
    ]
})

print(response["messages"][-1].content)

In [ ]:
messages = [
    {"role": "user", "content": f"Message {i}"}
    for i in range(11)
]

response = agent.invoke({
    "messages": messages
})

print(response["messages"][-1].content)

before_model
→ Do something BEFORE LLM call

after_model
→ Do something AFTER LLM call

dynamic_prompt
→ Change SYSTEM PROMPT dynamically

wrap_model_call
→ Control / modify the MODEL CALL itself

Model Router
→ Use wrap_model_call to select different models

In [ ]:
from langchain.agents.middleware import (
    SummarizationMiddleware, HumanInTheLoopMiddleware,
    ModelCallLimitMiddleware, ToolCallLimitMiddleware,
    ModelFallbackMiddleware, PIIMiddleware, TodoListMiddleware,
    LLMToolSelectorMiddleware, ToolRetryMiddleware, ModelRetryMiddleware,
    ContextEditingMiddleware, ClearToolUsesEdit,
)


SummarizationMiddleware → Automatically summarizes older conversation history when the context becomes too large.
HumanInTheLoopMiddleware → Pauses risky actions and asks a human to approve, edit, or reject them.
ModelCallLimitMiddleware → Limits how many times the LLM can be called to prevent runaway loops and high cost.
ToolCallLimitMiddleware → Limits how many times tools can be called during a run or thread.
ModelFallbackMiddleware → Uses another model automatically if the primary model fails.
PIIMiddleware → Detects and handles sensitive data such as emails, phone numbers, or credit-card details.
TodoListMiddleware → Lets the agent create and track a task/to-do list for multi-step work.
LLMToolSelectorMiddleware → Uses an LLM to select only the most relevant tools from a large tool set.
ToolRetryMiddleware → Automatically retries a tool when it fails because of a temporary error.
ModelRetryMiddleware → Automatically retries the model call when the LLM/API call fails temporarily.
ContextEditingMiddleware → Removes or edits old context to keep the conversation smaller and cleaner.
ClearToolUsesEdit → Removes older tool-call/tool-result messages from context while keeping recent ones.

!pip install -U deepagents

In [ ]:
from deepagents import create_deep_agent
from langchain.tools import tool


@tool
def search_web(query: str) -> str:
    """Search for information."""
    return f"Dummy search result for: {query}"


agent = create_deep_agent(
    model="openai:gpt-5.5",
    tools=[search_web],
    system_prompt="""
    You are a research assistant.
    Research the topic, organize the information,
    and provide a clear final report.
    """
)


result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Research Agentic AI and create a short report."
        }
    ]
})


print(result["messages"][-1].content)

User:
"Research Agentic AI and create a report"
        ↓
Deep Agent
        ↓
Plan the task
        ↓
Search information
        ↓
Maybe use subagents
        ↓
Store intermediate work/files
        ↓
Manage long context
        ↓
Produce final report

| Parameter | Simple meaning | When to use it | Example |
|---|---|---|---|
| `model` | The main LLM used by the agent | Set it explicitly in almost every agent | `model="openai:gpt-5.5"` |
| `tools` | Custom tools or APIs available to the agent | Use when the agent needs access to databases, search, calculators, APIs, RAG, etc. | `tools=[search_docs, get_weather]` |
| `system_prompt` | Main instructions, role, and behavior of the agent | Use when you want to define the agent’s role, rules, or task strategy | `system_prompt="You are a research assistant."` |
| `middleware` | Intercepts and customizes agent execution | Use for retries, logging, PII handling, limits, routing, or custom behavior | `middleware=[PIIMiddleware(...)]` |
| `subagents` | Specialized child agents | Use when research, coding, analysis, or other tasks should be delegated to specialists | `subagents=[researcher]` |
| `skills` | Reusable domain-specific instructions loaded from skill folders | Use for reusable expertise such as SQL, finance, security, or code review | `skills=["/skills/project/"]` |
| `memory` | Loads `AGENTS.md` memory files when the agent starts | Use when the agent should remember persistent instructions, preferences, or knowledge | `memory=["/memory/AGENTS.md"]` |
| `permissions` | Allow, deny, or approval rules for filesystem tools | Use when sensitive files or operations need access control | `permissions=[...]` |
| `backend` | Defines where files are stored and where execution happens | Use when you need local files, state storage, persistent storage, or sandboxed execution | `backend=FilesystemBackend(...)` |
| `interrupt_on` | Pauses before selected tool calls for human approval | Use for risky actions such as delete, send, edit, payment, or deployment | `interrupt_on={"send_email": True}` |
| `response_format` | Defines the exact structured format of the final output | Use when you need JSON, Pydantic, or another structured response | `response_format=Report` |
| `state_schema` | Defines custom internal state that can change while the agent runs | Use when the workflow needs custom changing fields or intermediate state | `state_schema=MyState` |
| `context_schema` | Defines fixed contextual information for one agent run | Use for data such as user ID, role, tenant, permissions, or environment | `context_schema=UserContext` |
| `checkpointer` | Saves thread or agent state between executions | Use when you need conversation memory, pause/resume, HITL, or workflow recovery | `checkpointer=MemorySaver()` |
| `store` | Long-term persistent storage | Use for cross-thread memory, user profiles, preferences, or persistent knowledge | `store=my_store` |
| `debug` | Enables debug information | Use during development and troubleshooting | `debug=True` |
| `name` | Gives the agent a name | Use to identify agents in multi-agent systems, tracing, and logs | `name="research_agent"` |
| `cache` | Reuses cached computations or results | Use when repeated computations or API calls can be safely reused to reduce cost and latency | `cache=my_cache` |

In [ ]:
agent = create_deep_agent(
    model=CHAT_MODEL,

    tools=[
        search_docs,
        search_database
    ],

    system_prompt="You are a senior research analyst.",

    subagents=[
        researcher,
        analyst
    ],

    skills=[
        "/skills/research/",
        "/skills/finance/"
    ],

    memory=[
        "/memory/AGENTS.md"
    ],

    backend=my_backend,

    permissions=my_permissions,

    interrupt_on={
        "send_email": True,
        "delete_file": True
    },

    response_format=ResearchReport,

    context_schema=UserContext,

    checkpointer=checkpointer,

    store=store,

    name="research_agent"
)

In [ ]:
# model          → WHO thinks
# tools          → WHAT actions it can perform
# system_prompt  → HOW it should behave
# skills         → WHAT expertise it can load
# memory         → WHAT it already knows/remembers
# subagents      → WHO it can delegate to
# backend        → WHERE it works
# permissions    → WHAT it is allowed to do
# interrupt_on   → WHAT needs human approval
# response_format→ HOW final answer must look
# state_schema   → WHAT changes during execution
# context_schema → WHO/WHAT this run belongs to
# checkpointer   → HOW current thread is saved
# store          → HOW long-term data is saved
# middleware     → HOW execution is controlled

In [ ]:
# """
# Complete Deep Agents demo.

# Install:
#     pip install -U deepagents langchain langgraph langchain-openai pydantic

# Before running:
#     Set OPENAI_API_KEY in your environment.
# """

# from dataclasses import dataclass
# from pathlib import Path

# from pydantic import BaseModel, Field

# from deepagents import create_deep_agent
# from deepagents.backends.filesystem import FilesystemBackend
# from deepagents.middleware.filesystem import FilesystemPermission

# from langchain.tools import tool
# from langgraph.checkpoint.memory import InMemorySaver
# from langgraph.store.memory import InMemoryStore


# # ============================================================
# # 1. MODEL
# # ============================================================

# CHAT_MODEL = "openai:gpt-5.4-mini"


# # ============================================================
# # 2. DUMMY CUSTOM TOOLS
# # ============================================================

# @tool
# def search_docs(query: str) -> str:
#     """Search internal documents."""
#     return (
#         f"Document result for '{query}': "
#         "LangChain is used for building LLM applications, "
#         "LangGraph provides low-level orchestration, and "
#         "Deep Agents is designed for complex long-running agent work."
#     )


# @tool
# def search_database(query: str) -> str:
#     """Search the internal company database."""
#     return (
#         f"Database result for '{query}': "
#         "Found 3 matching dummy records."
#     )


# @tool
# def send_email(to: str, subject: str, body: str) -> str:
#     """Send an email."""
#     return f"Dummy email sent to {to} with subject '{subject}'."


# @tool
# def delete_file(path: str) -> str:
#     """Delete a file."""
#     return f"Dummy delete completed for: {path}"


# # ============================================================
# # 3. SUBAGENTS
# # ============================================================

# researcher = {
#     "name": "researcher",
#     "description": "Researches technical topics using documents and database search.",
#     "system_prompt": """
# You are a research specialist.
# Research the given topic carefully.
# Use search_docs and search_database when useful.
# Return concise, factual findings to the parent agent.
# """,
#     "tools": [search_docs, search_database],
#     "model": CHAT_MODEL,
# }

# analyst = {
#     "name": "analyst",
#     "description": "Analyzes research findings and produces clear conclusions.",
#     "system_prompt": """
# You are a senior analyst.
# Analyze the information you receive.
# Identify the most important findings, comparisons, and conclusions.
# """,
#     "tools": [search_database],
#     "model": CHAT_MODEL,
# }


# # ============================================================
# # 4. DEMO WORKSPACE
# # ============================================================

# WORKSPACE = (Path.cwd() / "deep_agent_demo").resolve()
# WORKSPACE.mkdir(parents=True, exist_ok=True)


# # ============================================================
# # 5. SKILLS
# # ============================================================

# research_skill_dir = WORKSPACE / "skills" / "research"
# research_skill_dir.mkdir(parents=True, exist_ok=True)

# (research_skill_dir / "SKILL.md").write_text(
#     """---
# name: research
# description: Use this skill when performing technical research.
# ---

# # Research Skill

# 1. Understand the research question.
# 2. Search relevant documents.
# 3. Search the database if useful.
# 4. Compare findings.
# 5. Return evidence-based conclusions.
# """,
#     encoding="utf-8",
# )

# finance_skill_dir = WORKSPACE / "skills" / "finance"
# finance_skill_dir.mkdir(parents=True, exist_ok=True)

# (finance_skill_dir / "SKILL.md").write_text(
#     """---
# name: finance
# description: Use this skill when analyzing financial information.
# ---

# # Finance Skill

# 1. Check revenue.
# 2. Check expenses.
# 3. Identify trends.
# 4. Identify risks.
# 5. Provide a concise conclusion.
# """,
#     encoding="utf-8",
# )

# skills = ["/skills/"]


# # ============================================================
# # 6. MEMORY
# # ============================================================

# memory_dir = WORKSPACE / "memory"
# memory_dir.mkdir(parents=True, exist_ok=True)

# (memory_dir / "AGENTS.md").write_text(
#     """# Agent Memory

# - Prefer concise technical answers.
# - Use examples when explaining concepts.
# - The user is learning LangChain and Deep Agents.
# """,
#     encoding="utf-8",
# )

# memory = ["/memory/AGENTS.md"]


# # ============================================================
# # 7. BACKEND
# # ============================================================

# my_backend = FilesystemBackend(
#     root_dir=WORKSPACE,
#     virtual_mode=True,
# )


# # ============================================================
# # 8. FILESYSTEM PERMISSIONS
# # ============================================================

# my_permissions = [
#     FilesystemPermission(
#         operations=["write"],
#         paths=["/memory/**"],
#         mode="interrupt",
#     )
# ]


# # ============================================================
# # 9. STRUCTURED RESPONSE FORMAT
# # ============================================================

# class ResearchReport(BaseModel):
#     summary: str = Field(description="Short research summary")
#     key_findings: list[str] = Field(description="Important findings")
#     sources: list[str] = Field(description="Sources or tools used")
#     confidence: str = Field(description="Confidence level: low, medium, or high")


# # ============================================================
# # 10. RUN-SCOPED CONTEXT
# # ============================================================

# @dataclass
# class UserContext:
#     user_id: str
#     role: str


# # ============================================================
# # 11. CHECKPOINTER + STORE
# # ============================================================

# checkpointer = InMemorySaver()
# store = InMemoryStore()


# # ============================================================
# # 12. CREATE DEEP AGENT
# # ============================================================

# agent = create_deep_agent(
#     model=CHAT_MODEL,

#     tools=[
#         search_docs,
#         search_database,
#         send_email,
#         delete_file,
#     ],

#     system_prompt="""
# You are a senior research analyst.

# For complex research tasks:
# 1. Understand the task.
# 2. Use specialist subagents when useful.
# 3. Use skills when relevant.
# 4. Search documents or databases when needed.
# 5. Produce an evidence-based final report.

# Do not send emails or delete files unless the user explicitly asks.
# """,

#     subagents=[
#         researcher,
#         analyst,
#     ],

#     skills=skills,
#     memory=memory,

#     backend=my_backend,
#     permissions=my_permissions,

#     interrupt_on={
#         "send_email": True,
#         "delete_file": True,
#     },

#     response_format=ResearchReport,
#     context_schema=UserContext,
#     checkpointer=checkpointer,
#     store=store,
#     name="research_agent",
# )


# # ============================================================
# # 13. INVOKE
# # ============================================================

# def run_demo() -> None:
#     response = agent.invoke(
#         {
#             "messages": [
#                 {
#                     "role": "user",
#                     "content": """
# Research LangChain, LangGraph, and Deep Agents.

# Use the researcher and analyst if useful.

# Give me:
# - a short summary
# - key findings
# - sources used
# - confidence level

# Do not send any email.
# Do not delete any file.
# """,
#                 }
#             ]
#         },

#         context=UserContext(
#             user_id="101",
#             role="admin",
#         ),

#         config={
#             "configurable": {
#                 "thread_id": "research-thread-001",
#             }
#         },
#     )

#     print("\n=== STRUCTURED RESPONSE ===")
#     print(response.get("structured_response"))

#     print("\n=== FINAL MESSAGE ===")
#     if response.get("messages"):
#         print(response["messages"][-1].content)


# if __name__ == "__main__":
#     run_demo()


User
 ↓
UserContext
 ↓
Deep Agent
 │
 ├── System Prompt
 │
 ├── Memory
 │
 ├── Skills
 │      ├── Research
 │      └── Finance
 │
 ├── Tools
 │      ├── search_docs
 │      ├── search_database
 │      ├── send_email
 │      └── delete_file
 │
 ├── Subagents
 │      ├── Researcher
 │      └── Analyst
 │
 ├── FilesystemBackend
 │
 ├── Permissions
 │
 ├── HITL
 │
 ├── Checkpointer
 │
 └── Store
 ↓
ResearchReport

# LangSmith — Observability, Testing and Evals

# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = "..."
# os.environ["LANGSMITH_PROJECT"] = "langchain-master-course"

Trace: prompts, model calls, tools, retrieval, nested graphs, latency, tokens and errors.

Evaluate: correctness, groundedness, retrieval quality, tool selection/arguments, trajectory, task success, safety, latency and cost.

Testing docs include unit tests, integration tests and agent evaluations.